In [12]:
#conda install -p c:\Users\MAS203\Anaconda3\envs\back_sub_env ipykernel --update-deps --force-reinstall
#pip install pylabel
#pip install tqdm

In [29]:
yaml_path = '../../data/Wildcount/training_data/coco_camera_trap.json'
#yaml_path = '../../data/Wildcount/training_data/train_annotations_20211202.json'
image_path = '../../../Ecoflow_wildcount/working_environment/data/'
export_path = '../../data/Wildcount/training_data/labels'

In [24]:
os.listdir(image_path)

['annotated_image_summary.csv',
 'Example data wildcount',
 'Example_folder_structure',
 'images_to_be_transferred',
 'Image_record_20210820-094500.csv',
 'Image_record_20210820-094500_updated.xlsx',
 'image_record_annotates.csv',
 'non_wildcount_images.csv',
 'non_wildcount_images.xlsx',
 'pest_monitoring_images_eg.csv',
 'processed',
 'unique_non_wildcount_sequences.csv',
 '~$Image_record_20210820-094500_updated.xlsx',
 '~$non_wildcount_images.xlsx']

In [14]:
from pylabel import importer

In [26]:
dataset = importer.ImportCoco(path=yaml_path, path_to_images=image_path)


JSONDecodeError: Extra data: line 2 column 1 (char 337)

In [6]:
dataset.export.ExportToYoloV5(export_path)

Exporting YOLO files...: 100%|██████████| 44112/44112 [4:04:30<00:00,  3.01it/s]


['..\\..\\data\\Kakadu_fish\\training_data\\dataset.yaml',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\0.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\1.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\2.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\3.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\4.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\5.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\6.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\7.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\8.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\9.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\10.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\11.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\12.txt',
 '..\\..\\data\\Kakadu_fish\\training_data\\labels_all_2\\13.txt',
 '..\\..\\dat

In [31]:
#if the above doesnt work, let's create our own function

import json
import os

def coco_to_yolo(json_file_path, output_dir):
    with open(json_file_path, 'r') as f:
        data = json.load(f)

    images = data['images']
    annotations = data['annotations']
    categories = data['categories']

    for image in images:
        img_id = image['id']
        file_name = image['file_name']
        width = image['width']
        height = image['height']

        yolo_labels = []
        for ann in annotations:
            if ann['image_id'] == img_id:
                category_id = ann['category_id']
                category = next(cat for cat in categories if cat['id'] == category_id)
                category_name = category['name']

                bbox = ann['bbox']
                x_center = bbox[0] + bbox[2] / 2
                y_center = bbox[1] + bbox[3] / 2
                w = bbox[2]
                h = bbox[3]

                x_center /= width
                y_center /= height
                w /= width
                h /= height

                yolo_labels.append('{} {:.6f} {:.6f} {:.6f} {:.6f}'.format(
                    category_name, x_center, y_center, w, h
                ))

        if yolo_labels:
            txt_file_path = os.path.join(output_dir, os.path.splitext(file_name)[0].split('/')[-1] + '.txt')
            with open(txt_file_path, 'w') as f:
                f.write('\n'.join(yolo_labels))


This code takes as input the path to a COCO JSON file (json_file_path) and an output directory (output_dir) where the YOLO txt labels will be written. It assumes that the COCO JSON file contains the following keys: images, annotations, and categories.

The coco_to_yolo function first reads the COCO JSON file using the json module. It then extracts the relevant information from the images, annotations, and categories keys. For each image, it loops over all the annotations that belong to that image and converts the bounding box coordinates from COCO format to YOLO format. It then writes the YOLO labels to a txt file with the same name as the image file (but with a .txt extension) in the output directory.

Note that this code assumes that the category names in the COCO JSON file are valid YOLO class names (i.e., they contain no spaces or special characters). If your COCO JSON file contains category names that are not valid YOLO class names, you will need to modify the code to handle this.

In [32]:
coco_to_yolo(yaml_path,export_path)